In [14]:
import pandas as pd

# Your master panel (GDP + Attacks)
df_master = df_gdp_attacks_panel.copy()

# List of other datasets to merge
panel_datasets = [
    df_population_panel,
    df_poverty_panel,
    df_school_panel,
    df_unemployment,
    df_inequality_panel
]

# Step: Merge all datasets with suffixes to avoid column overlap
for i, df in enumerate(panel_datasets, start=1):
    # Add suffix based on dataset number or name
    suffix = f"_ds{i}"  # or use a descriptive name like "_pop", "_poverty"
    
    # Merge with suffix
    df_master = df_master.join(df, how='left', rsuffix=suffix)

# Optional: check merged panel
print("Master panel shape:", df_master.shape)
print("\nColumns in master panel:\n", df_master.columns)

# Step: Save to CSV
df_master.to_csv(r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\Final_Master_Panel_AllColumns.csv")

print("Master panel with all columns saved successfully!")


Master panel shape: (12066, 8)

Columns in master panel:
 Index(['Country Code', 'GDP', 'Attacks', 'Total_Population',
       'Poverty_Headcount', 'Secondary_Enrollment', 'Unemployment_Rate',
       'Inequality_Measure'],
      dtype='object')
Master panel with all columns saved successfully!


In [16]:
import pandas as pd

# ==========================
# Load WGI master panel
# ==========================
wgi = pd.read_excel(
    r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\WGI_master_panel.xlsx"
)
wgi.columns = wgi.columns.str.strip()
wgi["Country"] = wgi["Country"].str.strip().str.lower()

# ==========================
# Load terrorism data
# ==========================
attacks = pd.read_excel(
    r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\Terror.xlsx",
    sheet_name="Terror_Long_1970_2020"
)
attacks.columns = attacks.columns.str.strip()

# Standardize country name
attacks["Country"] = attacks["Country"].str.strip().str.lower()

# Convert attacks to numeric
attacks["Attacks"] = pd.to_numeric(attacks["Attacks"], errors="coerce")

# ==========================
# Drop redundant column
# ==========================
if "COUNTRIES" in attacks.columns:
    attacks = attacks.drop(columns=["COUNTRIES"])

# ==========================
# Safety check: one row per Country–Year
# ==========================
dup = attacks.duplicated(subset=["Country", "Year"]).sum()
print("Duplicate Country–Year rows in attacks:", dup)

# If duplicates exist, collapse (defensive)
if dup > 0:
    attacks = (
        attacks
        .groupby(["Country", "Year"], as_index=False)["Attacks"]
        .sum()
    )

# ==========================
# Merge with WGI panel
# ==========================
final_panel = wgi.merge(
    attacks,
    on=["Country", "Year"],
    how="left",
    validate="one_to_one"
)

# ==========================
# Final checks
# ==========================
print("Final panel shape:", final_panel.shape)
print("Countries:", final_panel["Country"].nunique())
print("Years:", final_panel["Year"].min(), "-", final_panel["Year"].max())
print(
    "Duplicate Country–Year rows after merge:",
    final_panel.duplicated(subset=["Country", "Year"]).sum()
)


Duplicate Country–Year rows in attacks: 0
Final panel shape: (3902, 9)
Countries: 166
Years: 1996 - 2024
Duplicate Country–Year rows after merge: 0


In [17]:
print(final_panel.head(10))


       Country  Year  Control of corruption  Rule of law  \
0  afghanistan  2004                    0.0        0.075   
1  afghanistan  2005                    0.0        0.075   
2  afghanistan  2006                    0.0        0.025   
3  afghanistan  2007                    NaN          NaN   
4  afghanistan  2008                    NaN          NaN   
5  afghanistan  2009                    0.0        0.075   
6  afghanistan  2010                    0.0        0.075   
7  afghanistan  2011                    0.0        0.075   
8  afghanistan  2012                    0.0        0.075   
9  afghanistan  2013                    0.0        0.075   

   Government Effectiveness  Political Stability  Voice and Accountability  \
0                      0.25               0.1875                  0.000000   
1                      0.25               0.0625                  0.000000   
2                      0.00               0.0625                  0.102000   
3                       NaN

In [19]:
output_path = r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\WGI_master_panel_final.xlsx"

final_panel.to_excel(
    output_path,
    sheet_name="WGI_Master_Panel",
    index=False
)

print(f"WGI master panel saved successfully at:\n{output_path}")


WGI master panel saved successfully at:
C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\WGI_master_panel_final.xlsx
